In [1]:
import pandas as pd


In [2]:
df=pd.read_csv("customer_shopping_behavior.csv")

In [3]:
df.shape

(3900, 18)

In [4]:
df.head(5)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [5]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [6]:
df['Review Rating']=df['Review Rating'].fillna(df['Review Rating'].median())


In [7]:
df.columns=df.columns.str.lower()

In [8]:
df.columns=df.columns.str.replace(' ','_')

In [9]:



df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [10]:
df=df.rename(columns={'purchase_amount_(usd)':'purchased_amount'})

In [11]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchased_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [12]:
#create a column age_group
labels=['Young Adults','Adults','Middle_age','Senior']
df['age_group']=pd.qcut(df['age'],q=4,labels=labels)


In [13]:
df[['age','age_group']].head(15)

,age,age_group
0,55,Middle_age
1,19,Young Adults
2,50,Middle_age
3,21,Young Adults
4,45,Middle_age
5,46,Middle_age
6,63,Senior
7,27,Young Adults
8,26,Young Adults
9,57,Middle_age


In [14]:
df.head(5)

,customer_id,age,gender,item_purchased,category,purchased_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,age_group
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,Middle_age
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly,Young Adults
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,Middle_age
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,Young Adults
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,Middle_age


In [15]:
#create column purchase_frequency_days
frequency_mapping={
    'Fortnightly':14,
    'Weekly':7,
     'Annually':365,
    'Quarterly':90,
    'Bi-Weekly':14,
    'Monthly':30,
    'Every 3 Months':90}
df['purchase_frequency_days']=df['frequency_of_purchases'].map(frequency_mapping)

In [16]:
df[['frequency_of_purchases','purchase_frequency_days']].head(10)

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7
8,Annually,365
9,Quarterly,90


In [17]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [18]:


pd.crosstab(df['discount_applied'],df['promo_code_used'])

promo_code_used,No,Yes
discount_applied,,
No,2223,0
Yes,0,1677


In [19]:
(df['discount_applied']==df['promo_code_used']).all()

np.True_

In [20]:
df=df.drop('promo_code_used',axis=1)

In [21]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchased_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

In [22]:
!pip install mysql-connector-python


In [23]:
import mysql.connector

In [25]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
username='postgres'
password='onkar@123'
host="localhost"
port='5432'
database='customer_behaviour'
password = quote_plus(password)

engine= create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

#step2 load dataframe into postgresql
table_name="customer" #choose any table name
df.to_sql("customer", engine, if_exists="replace", index=False)
print(f"Data successfully loaded into table '{table_name}' in database '{database}',")

Data successfully loaded into table 'customer' in database 'customer_behaviour',


In [26]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

username = "postgres"
password = "onkar@123"
host = "localhost"
port = 5432
database = "customer_behaviour"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{quote_plus(password)}@{host}:{port}/{database}"
)

# Test connection
with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


In [27]:
table_name = "customer"

df.to_sql(
    table_name,
    engine,
    if_exists="replace",
    index=False
)

print("Table created successfully!")

Table created successfully!
